# LatentMind V6 — Server Notebook

Loads the Djezzy assistant models **once**, then serves the API behind an ngrok
tunnel. No training/benchmark clutter — warm up and serve.

| # | Cell | What |
|---|------|------|
| 1 | Cleanup | (optional) free VRAM before a reload |
| 2 | Repo | mount Drive, hard-reset to latest `origin/main` |
| 3 | Deps | install inference + voice + server deps |
| 4 | Env | model size, paths, tokens, date anchor |
| 5 | Database | copy SQLite from Drive, make output dirs |
| 6 | Brain | one-time MLP train (cached on Drive after first run) |
| 7 | Models | brain + SLM (Qwen 4B) + polisher + BGE-M3 |
| 8 | Voice | faster-whisper STT + XTTS-v2 TTS |
| 9 | **Serve** | start server, **self-test the WebSocket**, open ngrok |

**Run top to bottom once.** After a code change, just rerun **cell 9** — it
stops the old server cleanly (no zombies) and reloads `v6.server` from disk.
Cell 9 prints `WS self-test: PASS ✓` when the server genuinely works; only then
paste the URL + token into the frontend.

In [ ]:
# ── CELL 1: GPU cleanup (optional) ───────────────────────────────────────────
# Run before reloading models after an OOM. Safe on a fresh runtime (no-op).
import gc, sys, torch

def cleanup(verbose=True, _globals=None):
    freed = []
    g = _globals or {}
    for name in ("agent", "slm", "stt", "tts"):
        if name in g:
            del g[name]; freed.append(name)
    try:
        import v6.slm as m
        if m._slm is not None:
            for tid in list(m._slm._store.keys()): m._slm.clear_thread(tid)
            if getattr(m._slm, "_draft", None) is not None: m._slm._draft = None
            if hasattr(m._slm, "model"): del m._slm.model
            m._slm = None; freed.append("SLM")
        if getattr(m, "_polisher", None) is not None:
            if hasattr(m._polisher, "model"): del m._polisher.model
            m._polisher = None; freed.append("polisher")
    except Exception: pass
    try:
        import v6.knowledge as m
        if getattr(m, "_encoder", None) is not None: m._encoder = None; freed.append("BGE-M3")
        m._retriever = None
    except Exception: pass
    try:
        import v6.brain as m
        if getattr(m, "_brain", None) is not None: m._brain = None; freed.append("brain")
    except Exception: pass
    try:
        import v6.speech as m
        if getattr(m, "_stt", None) is not None: m._stt = None; freed.append("STT")
        if getattr(m, "_tts", None) is not None: m._tts = None; freed.append("TTS")
    except Exception: pass
    for _ in range(3): gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
    if verbose:
        print("Freed:", ", ".join(freed) or "nothing")
        if torch.cuda.is_available():
            a = torch.cuda.memory_allocated() / 1e9
            t = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"VRAM: {a:.1f} / {t:.1f} GB")

print("cleanup() ready — call cleanup(_globals=globals()) to free VRAM")

In [ ]:
# ── CELL 2: Mount Drive + sync repo (hard reset to latest) ───────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, shutil

REPO_URL = 'https://github.com/Hamza09Hamza/Latent-Djezzy.git'
REPO_DIR = '/content/Latent-Djezzy'
BRANCH   = 'main'

os.chdir('/content')
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Syncing to latest origin/%s ...' % BRANCH)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '-q'], check=True)
    # hard reset guarantees the exact remote code (no stale local state)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard',
                    'origin/' + BRANCH, '-q'], check=True)
else:
    if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO_URL, REPO_DIR],
                   check=True)

# flush cached v6 modules so the synced code is what gets imported
for _m in list(sys.modules):
    if _m.startswith('v6'): del sys.modules[_m]
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short',
                                  'HEAD']).decode().strip()
print('Commit:', commit, ' | dir:', os.getcwd())

In [ ]:
# ── CELL 3: Install dependencies (one shot) ──────────────────────────────────
!pip install -q 'transformers>=4.46.0' 'sentence-transformers>=3.0.0' 'accelerate>=0.27.0'
!pip install -q 'langgraph>=0.2.0' 'bitsandbytes>=0.43.0' scipy matplotlib
!pip install -q jinja2 pydantic pymysql mysql-connector-python
print('inference + RAG OK')

!pip install -q faster-whisper soundfile rapidfuzz coqui-tts
!pip install -q 'transformers>=4.46.0'   # re-assert (coqui may downgrade)
print('voice (STT + TTS) OK')

!pip install -q fastapi uvicorn 'websockets>=12' python-multipart pyngrok nest-asyncio
print('server + tunnel OK')
print('\nAll dependencies installed')

In [ ]:
# ── CELL 4: Environment configuration ────────────────────────────────────────
import os, warnings, logging, datetime
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

os.environ['V6_USE_SQLITE']  = '1'
os.environ['V6_SQLITE_PATH'] = '/content/interndb.sqlite'

# Model: '4b' Qwen3-4B (~8GB, default) | '3b' lighter | '7b' needs V6_4BIT=1
os.environ['V6_SLM_SIZE']        = '4b'
os.environ['V6_4BIT']            = '0'
os.environ['V6_SPECULATIVE']     = '1'
os.environ['V6_CONSTRAINED_SQL'] = '0'
os.environ['V6_POLISHER_HUB_ID'] = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['V6_SLM_OVERRIDE']    = ''

os.environ['V6_OUTPUT_DIR']      = '/content/drive/MyDrive/LatentDjezzy/v6_output'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['V6_DATE_ANCHOR']     = datetime.date.today().isoformat()

# API token clients must send (frontend Settings + ?token= on the WebSocket)
os.environ['V6_API_TOKEN']       = 'djezzy-demo'

print('env OK  | SLM=%s 4bit=%s | date=%s | token=%s' % (
    os.environ['V6_SLM_SIZE'], os.environ['V6_4BIT'],
    os.environ['V6_DATE_ANCHOR'], os.environ['V6_API_TOKEN']))

In [ ]:
# ── CELL 5: Database + output directories ────────────────────────────────────
import shutil, os
LOCAL_DB = '/content/interndb.sqlite'
SRC = next((p for p in ['/content/drive/MyDrive/LatentDjezzy/interndb.sqlite',
                        '/content/drive/MyDrive/interndb.sqlite']
            if os.path.isfile(p)), None)
if not SRC:
    print('DB not found in Drive — checked LatentDjezzy/ and MyDrive root')
elif not os.path.isfile(LOCAL_DB):
    shutil.copy(SRC, LOCAL_DB)
    print('copied interndb.sqlite (%d bytes)' % os.path.getsize(LOCAL_DB))
else:
    print('SQLite already present')

base = '/content/drive/MyDrive/LatentDjezzy/v6_output'
for d in ('charts', 'emails', 'reports', 'audio'):
    os.makedirs(f'{base}/{d}', exist_ok=True)
print('output dirs ready:', base)

In [ ]:
# ── CELL 6: Train brain MLP (one-time, ~2 min; cached on Drive after) ────────
import os, shutil, sys, subprocess
LOCAL = '/content/Latent-Djezzy/models/brain_head.pt'
DRIVE = '/content/drive/MyDrive/LatentDjezzy/models/brain_head.pt'
os.makedirs(os.path.dirname(LOCAL), exist_ok=True)

if os.path.isfile(LOCAL):
    print('brain_head.pt present (%d bytes) — skip' % os.path.getsize(LOCAL))
elif os.path.isfile(DRIVE):
    shutil.copy(DRIVE, LOCAL)
    print('brain_head.pt restored from Drive (%d bytes)' % os.path.getsize(LOCAL))
else:
    print('training brain (~2 min)...')
    for step, mod in (('traces', 'v6.brain_data'), ('train', 'v6.train_brain')):
        r = subprocess.run([sys.executable, '-m', mod], cwd='/content/Latent-Djezzy',
                           capture_output=True, text=True)
        if r.returncode != 0:
            print('STDERR:', r.stderr[-2000:]); raise RuntimeError(mod + ' failed')
        print(step, 'done')
    os.makedirs(os.path.dirname(DRIVE), exist_ok=True)
    shutil.copy(LOCAL, DRIVE)
    print('saved brain_head.pt to Drive for next session')

In [ ]:
# ── CELL 7: Load brain + SLM + polisher + BGE-M3 ─────────────────────────────
import sys, torch
sys.path.insert(0, '/content/Latent-Djezzy')
try:
    cleanup(verbose=False, _globals=globals())
except NameError:
    pass

from v6.graph import LatentMindV6
from v6.slm import get_slm, get_polisher
from v6.brain import get_brain

print('loading BGE-M3 + SLM (Qwen 4B) — first run downloads ~8 GB ...')
agent = LatentMindV6()
get_slm()
get_brain()
print('loading polisher (Qwen 1.5B) ...')
try:
    get_polisher(); print('polisher ready')
except Exception as e:
    print('polisher unavailable (%s) — raw answers used' % e)

if torch.cuda.is_available():
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {a:.1f} / {t:.1f} GB  (free ~{t-a:.1f} GB)')

In [ ]:
# ── CELL 8: Load voice models (STT + TTS) ────────────────────────────────────
import os, torch
from v6.config import V6Config
from v6.speech import get_stt, get_tts

print('loading STT (faster-whisper large-v3) ...'); stt = get_stt()
print('loading TTS (XTTS-v2) ...');                 tts = get_tts()

print('voice ready — reference voices:')
for lang, label in (('fr', 'French'), ('en', 'English')):
    wav = V6Config.speaker_wav(lang)
    if wav and os.path.isfile(wav):
        print(f'  {label:8} clone {os.path.basename(wav)} ({os.path.getsize(wav)//1024} KB)')
    else:
        print(f'  {label:8} built-in voice')

if torch.cuda.is_available():
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {a:.1f} / {t:.1f} GB  (free ~{t-a:.1f} GB)')

In [ ]:
# ── CELL 9: SERVE  (start → self-test WebSocket → open ngrok) ────────────────
# Rerun anytime after a code change: it stops the old server cleanly (no
# zombies), reloads v6.server from disk, then self-tests before announcing.
import os, sys, time, socket, threading, asyncio, json
import nest_asyncio, uvicorn, websockets
from pyngrok import ngrok

NGROK_TOKEN = '3ERc2GB0MMOjOt2dleZiNurkxZl_55C3LUxsrrK7hH8qFtBx7'
TOKEN = os.environ.get('V6_API_TOKEN') or 'djezzy-demo'
os.environ['V6_API_TOKEN'] = TOKEN

# 1. stop a server started by a previous run of THIS cell
try:
    _srv.should_exit = True
    _srv_thread.join(timeout=6)
    print('stopped previous server')
except NameError:
    pass

# 2. reload only server/logic code; keep model singletons in VRAM
_keep = {'v6.slm', 'v6.brain', 'v6.speech', 'v6.knowledge'}
for _m in list(sys.modules):
    if _m.startswith('v6') and _m not in _keep:
        del sys.modules[_m]

# 3. wait until port 8000 is free
def _free(p):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', p)) != 0
for _ in range(40):
    if _free(8000): break
    time.sleep(0.25)
else:
    print('WARNING: port 8000 still busy — Restart runtime if self-test fails')

# 4. import fresh server + start uvicorn in a background thread
from v6.server import app, API_TOKEN
_cfg = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
_srv = uvicorn.Server(_cfg)
_srv.install_signal_handlers = lambda: None
_srv_thread = threading.Thread(target=_srv.run, daemon=True)
_srv_thread.start()
time.sleep(4)

# 5. SELF-TEST the WebSocket locally (the real proof the server works)
async def _selftest():
    try:
        async with websockets.connect(f'ws://localhost:8000/ws?token={API_TOKEN}') as w:
            await w.send(json.dumps({'question': 'hi', 'thread': 'selftest'}))
            async for raw in w:
                if json.loads(raw).get('type') == 'done':
                    return True
    except Exception as e:
        print('self-test error:', type(e).__name__, e)
        return False
nest_asyncio.apply()
_ok = await _selftest()
print('WS self-test:', 'PASS' if _ok else 'FAIL')

# 6. open ngrok only if the server is genuinely healthy
if _ok:
    ngrok.set_auth_token(NGROK_TOKEN)
    for _t in ngrok.get_tunnels():
        ngrok.disconnect(_t.public_url)
    URL = ngrok.connect(8000, 'http').public_url
    print('=' * 64)
    print('URL   :', URL)
    print('token :', API_TOKEN)
    print('WS    :', URL.replace('https', 'wss') + '/ws?token=' + API_TOKEN)
    print('=' * 64)
    print('Frontend Settings -> paste URL + token -> Save & connect.')
    print('Leave this cell running.')
else:
    print('Server not healthy — not opening ngrok.')
    print('Restart runtime and run cells 1-9 once.')